# Judge the Judge — MVP

Workshop harness for evaluating LLM-as-judge prompts on [LLMBar](https://github.com/princeton-nlp/LLMBar)
(Zeng et al., ICLR 2024).

**How it works:** you write a judge prompt; the harness runs it over a sample of
LLMBar pairs — every pair in **both presentation orders** — and reports:

1. **Accuracy** vs the gold labels (overall, and Natural vs Adversarial subsets —
   the adversarial gap is your judge's susceptibility to surface appeal)
2. **Position bias** — how often the verdict is consistent when the two responses
   are swapped, and which position wins when it isn't

The harness is identical in every round: only the prompt changes, so score changes
are attributable to the prompt. Estimated cost per run at the default settings:
well under $0.05 with `gpt-4.1-nano`.


## 1. Setup

In [ ]:
%pip install -q openai pandas matplotlib
import json, os, random, re, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

MODEL = "gpt-4.1-nano"
N_PER_SUBSET = 4    # pairs sampled per LLMBar subset (5 subsets -> 20 pairs, 40 API calls)
SEED = 42           # fixed so everyone in the room judges the same pairs
MAX_WORKERS = 5     # modest concurrency so 20 people sharing one key don't trip rate limits

In [ ]:
# API key: Colab secret -> environment variable -> paste prompt
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass
if not api_key:
    from getpass import getpass
    api_key = getpass("Paste the workshop OpenAI API key: ")

from openai import OpenAI
client = OpenAI(api_key=api_key)

## 2. Load the evaluation pairs

Each LLMBar instance is an instruction plus two responses, with a gold label for
which response objectively follows the instruction better. In the four
Adversarial subsets the *worse* response is deliberately more superficially
appealing.

In [ ]:
import subprocess

if not Path("LLMBar").exists():
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/princeton-nlp/LLMBar.git"], check=True)

SUBSETS = ["Natural", "Neighbor", "GPTInst", "GPTOut", "Manual"]
REQUIRED_KEYS = {"input", "output_1", "output_2", "label"}


def load_subset(name):
    # Locate the subset's json inside the clone without hard-coding the layout.
    candidates = [p for p in Path("LLMBar").rglob("*.json")
                  if p.parent.name == name and "Dataset" in p.parts]
    for path in sorted(candidates):
        items = json.loads(path.read_text())
        if isinstance(items, list) and items and REQUIRED_KEYS <= set(items[0]):
            return [{"subset": name, "pair_id": f"{name}-{i}",
                     "instruction": it["input"], "output_1": it["output_1"],
                     "output_2": it["output_2"], "gold": int(it["label"])}
                    for i, it in enumerate(items)]
    raise FileNotFoundError(f"could not find data for subset {name!r}")


rng = random.Random(SEED)
pairs = []
for name in SUBSETS:
    subset = load_subset(name)
    pairs += rng.sample(subset, min(N_PER_SUBSET, len(subset)))

print(f"{len(pairs)} pairs sampled:",
      pd.Series([p["subset"] for p in pairs]).value_counts().to_dict())

In [ ]:
# Eyeball one adversarial pair so you know what the judge is up against
ex = next(p for p in pairs if p["subset"] == "Neighbor")
print("INSTRUCTION:\n", ex["instruction"])
print("\nRESPONSE 1:\n", ex["output_1"][:600])
print("\nRESPONSE 2:\n", ex["output_2"][:600])
print("\nGOLD:", ex["gold"])

## 3. Write your judge prompt

Rules of the game:

- Use the placeholders `{instruction}`, `{response_1}`, `{response_2}` — the
  harness fills them in (and handles the order-swapping for you).
- Your prompt must make the model **end its reply with the number of the better
  response: `1` or `2`** — the harness parses the last standalone 1/2 in the reply.

In [ ]:
NAIVE_PROMPT = """You are comparing two responses to an instruction.

Instruction:
{instruction}

Response 1:
{response_1}

Response 2:
{response_2}

Which response is better? Reply with only the number 1 or 2."""

## 4. The harness

Identical in every round. Each pair is judged twice — original order and swapped —
so position bias is measured for free. Retries with exponential backoff handle
rate limits.

In [ ]:
def parse_verdict(text):
    """Last standalone 1 or 2 in the judge's reply, or None if unparseable."""
    found = re.findall(r"\b([12])\b", text)
    return int(found[-1]) if found else None


def call_judge(prompt_template, pair, flipped, retries=5):
    r1, r2 = pair["output_1"], pair["output_2"]
    if flipped:
        r1, r2 = r2, r1
    prompt = prompt_template.format(
        instruction=pair["instruction"], response_1=r1, response_2=r2)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL, temperature=0,
                messages=[{"role": "user", "content": prompt}])
            break
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt + random.random())
    text = resp.choices[0].message.content or ""
    pick_position = parse_verdict(text)   # what the judge saw on screen
    # map back to the underlying response number
    pick = None if pick_position is None else (3 - pick_position if flipped else pick_position)
    return {"pair_id": pair["pair_id"], "subset": pair["subset"], "flipped": flipped,
            "pick_position": pick_position, "pick": pick,
            "correct": None if pick is None else pick == pair["gold"],
            "prompt_tokens": resp.usage.prompt_tokens,
            "completion_tokens": resp.usage.completion_tokens,
            "raw": text}


def run_judge(prompt_template, pairs):
    jobs = [(p, flipped) for p in pairs for flipped in (False, True)]
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        records = list(ex.map(lambda j: call_judge(prompt_template, *j), jobs))
    return pd.DataFrame(records)

In [ ]:
PRICE_IN, PRICE_OUT = 0.10, 0.40   # $ per 1M tokens, gpt-4.1-nano


def summarize(df):
    correct = df["correct"].astype("boolean")
    by_order = df.pivot(index="pair_id", columns="flipped", values="pick")
    consistent = by_order[False] == by_order[True]
    # An inconsistent pair means the judge stuck to a *position* across the swap;
    # this measures which position it stuck to.
    sticky = df[df.pair_id.isin(consistent[~consistent].index)]
    first_share = (sticky["pick_position"] == 1).mean() if len(sticky) else float("nan")
    return {
        "accuracy": correct.mean(),
        "accuracy_natural": correct[df.subset == "Natural"].mean(),
        "accuracy_adversarial": correct[df.subset != "Natural"].mean(),
        "positional_consistency": consistent.mean(),
        "first_position_share_when_inconsistent": first_share,
        "parse_failures": int(df["pick"].isna().sum()),
        "cost_usd": (df.prompt_tokens.sum() * PRICE_IN
                     + df.completion_tokens.sum() * PRICE_OUT) / 1e6,
    }


def subset_accuracy(df):
    return (df.assign(c=df["correct"].astype("boolean"))
              .groupby("subset")["c"].mean().reindex(SUBSETS))


RUNS = {}


def evaluate(name, prompt_template):
    df = run_judge(prompt_template, pairs)
    RUNS[name] = df
    stats = summarize(df)
    print(f"=== {name} ===")
    for k, v in stats.items():
        print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")
    return stats

## 5. Round 1 — run the naive judge

In [ ]:
evaluate("naive", NAIVE_PROMPT)

## 6. Results

Accuracy by subset. The dashed line is coin-flip chance — on adversarial subsets a
naive judge often lands near (or below) it.

In [ ]:
import matplotlib.pyplot as plt

SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # fixed slot order, never cycled


def plot_runs(names=None):
    names = (names or list(RUNS))[:4]   # >4 series stops being readable — facet instead
    fig, ax = plt.subplots(figsize=(8, 4), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    group = 0.8
    width = group / len(names)
    for i, name in enumerate(names):
        acc = subset_accuracy(RUNS[name])
        xs = [j - group / 2 + (i + 0.5) * width for j in range(len(SUBSETS))]
        ax.bar(xs, acc.values, width * 0.9, color=SERIES[i], label=name, zorder=3)
        for x, v in zip(xs, acc.values):
            if pd.notna(v):
                ax.text(x, v + 0.02, f"{v:.0%}", ha="center", fontsize=8, color=INK)
    ax.axhline(0.5, color=MUTED, linewidth=1, linestyle=(0, (4, 3)), zorder=2)
    ax.text(len(SUBSETS) - 0.55, 0.515, "chance", color=MUTED, fontsize=8)
    ax.set_xticks(range(len(SUBSETS)), SUBSETS, color=MUTED)
    ax.set_ylim(0, 1.08)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0],
                  ["0%", "25%", "50%", "75%", "100%"], color=MUTED)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(color=GRID, labelcolor=MUTED)
    if len(names) > 1:
        ax.legend(frameon=False, loc="upper right", labelcolor=INK)
    ax.set_title("Judge accuracy by subset", color=INK, loc="left")
    plt.tight_layout()
    plt.show()


plot_runs()

In [ ]:
# Read the judge's own words on the pairs it got wrong — often the most
# convincing exhibit: watch it praise the polished answer that ignores the task.
wrong = RUNS["naive"].query("correct == False")
for _, row in wrong.head(2).iterrows():
    p = next(p for p in pairs if p["pair_id"] == row.pair_id)
    print("INSTRUCTION:", p["instruction"][:200])
    print("JUDGE SAID:", row.raw[:200])
    print("(gold was", p["gold"], "— judge picked", row.pick, ")\n")

## 7. Round 2 — apply the theory, judge again

Rewrite the prompt using what the research says actually works (LLMBar paper,
Table 3): explicit **rules** that prioritize instruction-following over style,
self-generated **metrics**, or a self-generated **reference** answer. Plain
"think step by step" does *not* help — the reasoning drifts toward the prettier
answer and rationalizes it.

Below is one example to verify the mechanics — in the workshop, participants
write their own.

In [ ]:
RULES_PROMPT = """You are comparing two responses to an instruction. Select the \
response that better follows the instruction.

Evaluation rules:
1. First identify precisely what the instruction asks for, including any \
constraints or required content.
2. A response that does exactly what was asked beats one that is longer, \
friendlier, or better formatted but misses or ignores any part of the instruction.
3. Do not reward extra information that was not requested.
4. Instruction compliance and accuracy outweigh style, tone, and length.

Instruction:
{instruction}

Response 1:
{response_1}

Response 2:
{response_2}

In one sentence, state what the instruction requires. Then end your reply with \
the number of the better response: 1 or 2."""

evaluate("rules", RULES_PROMPT)
plot_runs()

## Next steps (not in this MVP)

- **Verbosity metric**: padded variants of wrong answers at increasing lengths —
  dose-response curve of accuracy vs length ratio
- **Self-preference metric**: flawed answers restyled by the judge's own model vs
  another model, same content
- **Shared leaderboard**: final cell POSTs each participant's metrics to a Google
  Sheet via an Apps Script webhook
- Larger sample (`N_PER_SUBSET = 8`) for steadier numbers on the day